# Module 08: Multiple Plots and Layouts
**CHIRPS Rainfall Data – Ethiopia**

Combine multiple views of the same dataset in a single figure using subplots,
GridSpec, inset axes, twin axes, and automatic layout management.



In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.dates import YearLocator, YearFormatter

ds = xr.open_dataset(r'../chirps_1981_2022.nc', engine='netcdf4')
ts = ds.precip.isel(latitude=180, longitude=174)
ts_df = pd.DataFrame({'precip': ts.values}, index=pd.DatetimeIndex(ts.time.values))
monthly_mean = ts_df.groupby(ts_df.index.month)['precip'].mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print("Data loaded")



: 

---
## 8.1 Basic Subplots with `plt.subplots`

The simplest way to create multiple panels is:
`fig, axes = plt.subplots(nrows, ncols, figsize=(width, height))`

`axes` is a 2D array of Axes objects.



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Full time series
axes[0,0].plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
axes[0,0].set_title('Full Time Series')
axes[0,0].set_ylabel('mm/month')

# Zoom 2000–2010
axes[0,1].plot(ts_df.index, ts_df['precip'], linewidth=0.8, color='coral')
axes[0,1].set_xlim(pd.Timestamp('2000-01-01'), pd.Timestamp('2010-12-01'))
axes[0,1].set_title('2000–2010 Zoom')
axes[0,1].set_ylabel('mm/month')

# Climatology
axes[1,0].bar(months, monthly_mean, color='mediumseagreen')
axes[1,0].set_title('Monthly Climatology')
axes[1,0].set_ylabel('mm/month')
axes[1,0].tick_params(axis='x', rotation=45)

# Histogram
axes[1,1].hist(ts_df['precip'], bins=40, color='steelblue', edgecolor='white')
axes[1,1].set_title('Rainfall Distribution')
axes[1,1].set_xlabel('mm/month')
axes[1,1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()



---
## 8.2 GridSpec for Complex Layouts

`GridSpec` gives you fine control over subplot placement, including
spans (rows/columns that cover multiple cells), varying sizes, and
custom positioning.



In [ ]:
fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3,
                       width_ratios=[2, 1, 1], height_ratios=[1, 1, 1.5])

# Row 0 spans: time series across whole top
ax0 = fig.add_subplot(gs[0, :])
ax0.plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
ax0.set_title('CHIRPS Addis Ababa – Full Record (1981–2022)')
ax0.set_ylabel('mm/month')

# Row 1: climatology bar, histogram, seasonal pie
ax1 = fig.add_subplot(gs[1, 0])
ax1.bar(months, monthly_mean, color='royalblue')
ax1.set_title('Climatology')
ax1.tick_params(axis='x', rotation=45)

ax2 = fig.add_subplot(gs[1, 1])
ax2.hist(ts_df['precip'], bins=30, color='mediumseagreen', edgecolor='white')
ax2.set_title('Distribution')
ax2.set_xlabel('mm/month')

ts_df['season'] = ts_df.index.month.map({12:'DJF',1:'DJF',2:'DJF',
                                          3:'MAM',4:'MAM',5:'MAM',
                                          6:'JJA',7:'JJA',8:'JJA',
                                          9:'SON',10:'SON',11:'SON'})
seasonal = ts_df.groupby('season')['precip'].sum()[['DJF','MAM','JJA','SON']]
ax3 = fig.add_subplot(gs[1, 2])
ax3.pie(seasonal, labels=seasonal.index, autopct='%1.0f%%')
ax3.set_title('Seasonal')

# Row 2: map spans 2 cols, 1 col for scatter
lat_slice = ds.precip.isel(time=0)
ax4 = fig.add_subplot(gs[2, :2])
im = ax4.pcolormesh(ds.longitude, ds.latitude, lat_slice, cmap='Blues', shading='auto')
ax4.set_title('CHIRPS – January 1981 Rainfall')
ax4.set_xlabel('Longitude'); ax4.set_ylabel('Latitude')
plt.colorbar(im, ax=ax4, label='mm/month', shrink=0.8)

ax5 = fig.add_subplot(gs[2, 2])
precip = ts_df['precip'].values
ax5.scatter(precip[:-1], precip[1:], s=5, alpha=0.3, color='darkgreen')
ax5.plot([0, precip.max()], [0, precip.max()], 'r--', linewidth=0.8)
ax5.set_title('Lag-1 Scatter')
ax5.set_xlabel('t (mm)'); ax5.set_ylabel('t+1 (mm)')
ax5.set_aspect('equal')

plt.tight_layout()
plt.show()



---
## 8.3 Shared Axes (`sharex`, `sharey`)

Sharing axes ensures that subplots have identical scales, making
comparison easier. Use `sharex=True` / `sharey=True` or `'row'` / `'col'`.



In [ ]:
# Shared x-axis – compare two locations
gondar = ds.precip.isel(latitude=252, longitude=150)
gondar_df = pd.DataFrame({'precip': gondar.values}, index=pd.DatetimeIndex(gondar.time.values))

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, sharey=True)

axes[0].plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
axes[0].set_ylabel('Addis (mm/month)')
axes[0].axhline(ts_df['precip'].mean(), color='steelblue', linestyle='--', linewidth=0.5)

axes[1].plot(gondar_df.index, gondar_df['precip'], linewidth=0.5, color='darkorange')
axes[1].set_ylabel('Gondar (mm/month)')
axes[1].axhline(gondar_df['precip'].mean(), color='darkorange', linestyle='--', linewidth=0.5)
axes[1].set_xlabel('Date')

fig.suptitle('Shared X and Y Axes – Addis Ababa vs Gondar', fontsize=13)
plt.tight_layout()
plt.show()



In [ ]:
# sharex='col', sharey='row'
locations = {
    'Addis Ababa': (180, 174),
    'Gondar':      (252, 150),
    'Mekelle':     (270, 190),
}

fig, axes = plt.subplots(len(locations), 2, figsize=(14, 9),
                         sharex='col', sharey='col')

for i, (name, (li, lo)) in enumerate(locations.items()):
    serie = ds.precip.isel(latitude=li, longitude=lo)
    ts_loc = pd.DataFrame({'precip': serie.values}, index=pd.DatetimeIndex(serie.time.values))
    clim = ts_loc.groupby(ts_loc.index.month)['precip'].mean()

    # Time series (column 0)
    axes[i,0].plot(ts_loc.index, ts_loc['precip'], linewidth=0.4)
    axes[i,0].set_ylabel(f'{name}\n(mm/month)', fontsize=9)

    # Climatology (column 1)
    axes[i,1].bar(months, clim, color='steelblue', alpha=0.7)
    axes[i,1].tick_params(axis='x', rotation=45)

axes[0,0].set_title('Time Series'); axes[0,1].set_title('Climatology')
axes[-1,0].set_xlabel('Date'); axes[-1,1].set_xlabel('Month')
plt.tight_layout()
plt.show()



---
## 8.4 Inset Plots

Inset plots place a smaller axes inside the main axes, useful for
zooming into a region of interest or showing detail.
Use `ax.inset_axes([x0, y0, width, height])` in figure coordinates.



In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Main plot – full time series
ax.plot(ts_df.index, ts_df['precip'], linewidth=0.6, color='steelblue')
ax.set_title('CHIRPS Addis Ababa – Full Record with Inset Zoom')
ax.set_ylabel('Precipitation (mm/month)')
ax.set_xlabel('Date')

# Inset – zoom on 1997 (strong El Niño year)
ax_inset = ax.inset_axes([0.15, 0.55, 0.25, 0.35])
year97 = ts_df.loc['1997']
ax_inset.plot(year97.index, year97['precip'], 'o-', color='coral', markersize=4)
ax_inset.set_title('1997 (El Niño)', fontsize=10)
ax_inset.set_ylabel('mm/month', fontsize=8)
ax_inset.tick_params(labelsize=7)

# Mark the zoomed region
ax.indicate_inset_zoom(ax_inset, edgecolor='gray')

plt.tight_layout()
plt.show()



In [ ]:
# Inset showing the histogram alongside a time series
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(ts_df.index, ts_df['precip'], linewidth=0.6, color='steelblue')
ax.set_title('Time Series with Embedded Histogram')
ax.set_ylabel('Precipitation (mm/month)')

# Inset histogram
ax_hist = ax.inset_axes([0.72, 0.55, 0.25, 0.35])
ax_hist.hist(ts_df['precip'], bins=30, color='mediumseagreen', edgecolor='white', orientation='horizontal')
ax_hist.set_title('Distribution', fontsize=9)
ax_hist.set_xlabel('Count', fontsize=8)
ax_hist.tick_params(labelsize=7)
ax_hist.axhline(ts_df['precip'].mean(), color='red', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.show()



---
## 8.5 Twin Axes

`ax.twinx()` and `ax.twiny()` create a second axes that shares the
same x or y axis but with an independent scale on the opposite side.



In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

# Primary: monthly rainfall
ax1.bar(months, monthly_mean, color='steelblue', alpha=0.6, label='Mean rainfall')
ax1.set_ylabel('Precipitation (mm/month)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Secondary: coefficient of variation
monthly_std = ts_df.groupby(ts_df.index.month)['precip'].std()
monthly_cv = monthly_std / monthly_mean * 100  # CV in %

ax2 = ax1.twinx()
ax2.plot(months, monthly_cv, 'o-', color='darkred', linewidth=2, markersize=6, label='CV (%)')
ax2.set_ylabel('Coefficient of Variation (%)', color='darkred')
ax2.tick_params(axis='y', labelcolor='darkred')

ax1.set_xlabel('Month')
ax1.set_title('Twinx: Mean Rainfall (bars) vs Coefficient of Variation (line)')
ax1.tick_params(axis='x', rotation=45)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()



---
## 8.6 Layout Management

Matplotlib offers several layout engines:
- `plt.tight_layout()` – adjusts subplot params automatically
- `constrained_layout=True` – set when creating the figure; more robust
- `figure.subplots_adjust()` – manual control (left, right, top, bottom, wspace, hspace)



In [ ]:
# Compare tight_layout vs constrained_layout vs no layout

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    ax.plot(ts_df.index[:120], ts_df['precip'][:120])
    ax.set_title(f'Panel {i+1}', fontsize=14)
    ax.set_xlabel('Long label that might get cut off')
    ax.set_ylabel('Precipitation (mm/month)')

plt.tight_layout(pad=2.0)
fig.suptitle('tight_layout(pad=2.0)', y=1.02, fontsize=13)
plt.show()



In [ ]:
# constrained_layout
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for i, ax in enumerate(axes.flat):
    ax.plot(ts_df.index[:120], ts_df['precip'][:120])
    ax.set_title(f'Panel {i+1}', fontsize=14)
    ax.set_xlabel('This label will not get cut off')
    ax.set_ylabel('Precipitation (mm/month)')

fig.suptitle('constrained_layout=True', fontsize=13)
plt.show()



In [ ]:
# Manual subplots_adjust
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    ax.plot(ts_df.index[:120], ts_df['precip'][:120])
    ax.set_title(f'Panel {i+1}')
    ax.set_xlabel('X label'); ax.set_ylabel('Y label')

fig.subplots_adjust(left=0.1, right=0.95, top=0.92, bottom=0.1, wspace=0.25, hspace=0.3)
fig.suptitle('Manual subplots_adjust', fontsize=13, y=0.98)
plt.show()



---
## 8.7 Multi-Panel CHIRPS Analysis Figure

A real-world example combining multiple views of CHIRPS data:
map + time series + histogram + seasonal breakdown.



In [ ]:
# Extract data for several locations
locations = {
    'Addis Ababa': (180, 174),
    'Gondar':      (252, 150),
    'Mekelle':     (270, 190),
}

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 3, figure=fig, height_ratios=[1.2, 1, 1],
                       width_ratios=[1.5, 1, 1], hspace=0.3, wspace=0.3)

# (0,0) – Map of mean rainfall
ax_map = fig.add_subplot(gs[0, 0])
mean_precip = ds.precip.mean(dim='time')
im = ax_map.pcolormesh(ds.longitude, ds.latitude, mean_precip,
                       cmap='YlGnBu', shading='auto')
ax_map.set_title('Mean CHIRPS Rainfall (1981–2022)', fontsize=11)
ax_map.set_xlabel('Longitude'); ax_map.set_ylabel('Latitude')
ax_map.set_aspect(1.0 / np.cos(np.deg2rad(9)))
cbar = plt.colorbar(im, ax=ax_map, shrink=0.7)
cbar.set_label('mm/month')

# Mark locations on map
colours = ['darkred', 'darkorange', 'darkgreen']
for j, (name, (li, lo)) in enumerate(locations.items()):
    lon_val = float(ds.longitude[lo])
    lat_val = float(ds.latitude[li])
    ax_map.plot(lon_val, lat_val, marker='D', color=colours[j], markersize=6)
    ax_map.text(lon_val + 0.5, lat_val, name, fontsize=8, color=colours[j])

# (0,1:3) – Time series for all locations
ax_ts = fig.add_subplot(gs[0, 1:])
for j, (name, (li, lo)) in enumerate(locations.items()):
    serie = ds.precip.isel(latitude=li, longitude=lo)
    ax_ts.plot(serie.time, serie, linewidth=0.4, color=colours[j], label=name, alpha=0.8)
ax_ts.set_title('Monthly Rainfall Comparison', fontsize=11)
ax_ts.set_ylabel('Precipitation (mm/month)')
ax_ts.legend(fontsize=9, ncol=3)
ax_ts.xaxis.set_major_locator(YearLocator(5))
ax_ts.xaxis.set_major_formatter(YearFormatter())

# (1,0) – Histogram Addis
ax_hist = fig.add_subplot(gs[1, 0])
ax_hist.hist(ts_df['precip'], bins=40, color='steelblue', edgecolor='white')
ax_hist.axvline(ts_df['precip'].mean(), color='red', ls='--')
ax_hist.set_title('Addis Rainfall Distribution', fontsize=10)
ax_hist.set_xlabel('mm/month'); ax_hist.set_ylabel('Count')

# (1,1) – Climatology Addis
ax_clim = fig.add_subplot(gs[1, 1])
ax_clim.bar(months, monthly_mean, color='steelblue')
ax_clim.set_title('Addis Climatology', fontsize=10)
ax_clim.tick_params(axis='x', rotation=45, labelsize=8)

# (1,2) – Seasonal pie Addis
ax_pie = fig.add_subplot(gs[1, 2])
seasonal = ts_df.groupby('season')['precip'].sum()[['DJF','MAM','JJA','SON']]
ax_pie.pie(seasonal, labels=seasonal.index, autopct='%1.0f%%',
           colors=['cornflowerblue','lightgreen','gold','lightcoral'])
ax_pie.set_title('Addis Seasonal Split', fontsize=10)

# (2,0) – Gondar vs Addis scatter
gondar_ts = ds.precip.isel(latitude=252, longitude=150).values
ax_scatter = fig.add_subplot(gs[2, 0])
ax_scatter.scatter(gondar_ts, ts_df['precip'].values, s=4, alpha=0.2, color='darkgreen')
ax_scatter.set_aspect('equal')
lim = [0, max(gondar_ts.max(), ts_df['precip'].max())]
ax_scatter.set_xlim(lim); ax_scatter.set_ylim(lim)
ax_scatter.plot(lim, lim, 'r--', linewidth=0.5)
ax_scatter.set_xlabel('Gondar (mm/month)')
ax_scatter.set_ylabel('Addis (mm/month)')
ax_scatter.set_title('Gondar vs Addis', fontsize=10)

# (2,1) – Mekelle time series 2015-2020
mekelle = ds.precip.isel(latitude=270, longitude=190)
mekelle_df = pd.DataFrame({'precip': mekelle.values}, index=pd.DatetimeIndex(mekelle.time.values))
sub_mek = mekelle_df.loc['2015':'2020']
ax_mek = fig.add_subplot(gs[2, 1])
ax_mek.plot(sub_mek.index, sub_mek['precip'], 'o-', color='darkorange', markersize=2, linewidth=0.6)
ax_mek.set_title('Mekelle 2015–2020', fontsize=10)
ax_mek.set_xlabel('Date'); ax_mek.set_ylabel('mm/month')

# (2,2) – CV bar chart
monthly_std_all = ts_df.groupby(ts_df.index.month)['precip'].std()
cv = monthly_std_all / monthly_mean * 100
ax_cv = fig.add_subplot(gs[2, 2])
ax_cv.bar(months, cv, color='lightcoral')
ax_cv.set_title('Addis CV (%)', fontsize=10)
ax_cv.set_ylabel('CV (%)'); ax_cv.tick_params(axis='x', rotation=45, labelsize=8)

fig.suptitle('CHIRPS Ethiopia – Multi-Panel Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.show()



---
## Exercises – Module 08

1. **Subplots basic**: Create a 3×1 figure showing the time series for **Addis Ababa**,
   **Gondar**, and **Mekelle** stacked vertically with shared x-axis.

2. **GridSpec**: Create a figure with a map of mean CHIRPS rainfall on the left (spanning
   full height) and three time series panels stacked on the right.

3. **Shared axes**: Plot monthly climatologies for 4 Ethiopian cities in a 2×2 grid with
   shared y-axis (`sharey='row'` or `sharey=True`).

4. **Inset plots**: Create a time series for Addis Ababa (1981–2022) with an inset that
   zooms into the 1984–1985 drought period.

5. **Twin axes**: Create a plot showing **mean rainfall** (bars) and **number of wet months
   > 100 mm** (line) per decade. Use `twinx`.

6. **Layout management**: Create a 3×3 subplot figure and compare `tight_layout(pad=1)`,
   `constrained_layout=True`, and `subplots_adjust(hspace=0.4, wspace=0.4)`.

### Mini-Project: Comprehensive CHIRPS Dashboard

Create a dashboard-style figure with **4 panels** using `GridSpec`:
- **Panel 1** (top, full width): Map of mean annual CHIRPS rainfall with location markers
  for 5 Ethiopian cities. Include a colourbar.
- **Panel 2** (bottom-left): Time series for two cities (Addis + one of your choice) on
  the same axes with a legend.
- **Panel 3** (bottom-centre): Side-by-side monthly climatologies (grouped bar chart) for
  the same two cities.
- **Panel 4** (bottom-right): Histogram of monthly rainfall for both cities with
  transparent overlapping bars.

Use `constrained_layout=True`, professional styling ('seaborn-v0_8'), and
include proper labels, titles, and colour coding consistent across panels.



In [ ]:
# Mini-Project starter
import matplotlib.patches as mpatches

plt.style.use('seaborn-v0_8')

fig = plt.figure(figsize=(16, 12), constrained_layout=True)
gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1.2, 1])

# Panel 1 – Map (spans top row)
ax_map = fig.add_subplot(gs[0, :])
mean_precip = ds.precip.mean(dim='time')
im = ax_map.pcolormesh(ds.longitude, ds.latitude, mean_precip,
                       cmap='YlGnBu', shading='auto')
ax_map.set_aspect(1.0 / np.cos(np.deg2rad(9)))
ax_map.set_title('Mean CHIRPS Rainfall (1981–2022)', fontsize=13, fontweight='bold')
ax_map.set_xlabel('Longitude'); ax_map.set_ylabel('Latitude')
cbar = plt.colorbar(im, ax=ax_map, shrink=0.6)
cbar.set_label('Precipitation (mm/month)')

# Location data
city_data = [
    ('Addis Ababa', 180, 174, 'darkred'),
    ('Gondar', 252, 150, 'darkorange'),
    ('Mekelle', 270, 190, 'darkgreen'),
    ('Dire Dawa', 192, 237, 'purple'),
    ('Jimma', 8, 144, 'teal'),
]

for name, li, lo, col in city_data:
    lon_val, lat_val = float(ds.longitude[lo]), float(ds.latitude[li])
    ax_map.plot(lon_val, lat_val, 'D', color=col, markersize=7, zorder=5)
    ax_map.text(lon_val + 0.4, lat_val, name, fontsize=9, color=col, fontweight='bold', zorder=5)

# Panel 2 – Time series (bottom-left)
ax_ts = fig.add_subplot(gs[1, 0])
for name, li, lo, col in city_data[:2]:
    serie = ds.precip.isel(latitude=li, longitude=lo)
    ax_ts.plot(serie.time, serie, linewidth=0.4, color=col, label=name, alpha=0.8)
ax_ts.set_title('Monthly Rainfall Comparison', fontsize=11)
ax_ts.set_xlabel('Date'); ax_ts.set_ylabel('Precipitation (mm/month)')
ax_ts.legend(fontsize=9)
ax_ts.xaxis.set_major_locator(YearLocator(5))

# Panel 3 – Grouped bar (bottom-centre)
ax_bar = fig.add_subplot(gs[1, 1])
width = 0.35
x = np.arange(12)
for i, (name, li, lo, col) in enumerate(city_data[:2]):
    serie = ds.precip.isel(latitude=li, longitude=lo)
    clim = serie.groupby('time.month').mean()
    ax_bar.bar(x + i*width, clim, width, color=col, label=name, alpha=0.8)
ax_bar.set_xticks(x + width/2)
ax_bar.set_xticklabels(months, rotation=45, fontsize=8)
ax_bar.set_title('Monthly Climatology', fontsize=11)
ax_bar.set_ylabel('Precipitation (mm/month)')
ax_bar.legend(fontsize=9)

# Panel 4 – Histogram (bottom-right)
ax_hist = fig.add_subplot(gs[1, 2])
for name, li, lo, col in city_data[:2]:
    serie = ds.precip.isel(latitude=li, longitude=lo)
    ax_hist.hist(serie.values, bins=40, alpha=0.5, color=col, label=name)
ax_hist.set_title('Rainfall Distribution', fontsize=11)
ax_hist.set_xlabel('Precipitation (mm/month)')
ax_hist.set_ylabel('Frequency')
ax_hist.legend(fontsize=9)

plt.show()
plt.style.use('default')

